In [0]:
dbutils.widgets.removeAll()

In [0]:
dbutils.widgets.text("reconciliation_table", "oh_apm_stg.tmp.ctl_reconciliation_log_Elig")
dbutils.widgets.text("error_table", "oh_apm_stg.vendor_extracts.error_load_report_log_cpc_Stg_Elig")
dbutils.widgets.text("columns_to_check", "SAK_RECIP")
dbutils.widgets.text("s3_path", "s3://gia-stg-oh-ue1-data-raw/haven/inbound/VE_EDW/weekly")

In [0]:
reconciliation_table = dbutils.widgets.get("reconciliation_table")
error_table = dbutils.widgets.get("error_table")
columns_to_check = dbutils.widgets.get("columns_to_check").split(",")
s3_path = dbutils.widgets.get("s3_path")

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, isnan, count
from datetime import datetime
from pyspark.sql.types import StructType, StructField, StringType, LongType
import ast

def to_bool(val):
    if isinstance(val, bool):
        return val
    if isinstance(val, str):
        return val.strip().lower() == "true"
    return False

try:
    critical_error_flag_raw = dbutils.jobs.taskValues.get(taskKey="Pre_validation", key="critical_error_flag", debugValue=None)
    soft_error_flag_raw = dbutils.jobs.taskValues.get(taskKey="Pre_validation", key="soft_error_flag", debugValue=None)

    print(f"🔎 Retrieved flags: critical_error_flag={critical_error_flag_raw}, soft_error_flag={soft_error_flag_raw}")

    # Convert to proper booleans
    critical_error_flag = to_bool(critical_error_flag_raw)
    soft_error_flag = to_bool(soft_error_flag_raw)
    print(f"✅ Converted flags: critical={critical_error_flag}, soft={soft_error_flag}")

except Exception as e:
    print(f"❌ Error retrieving task values: {e}")
    critical_error_flag = False
    soft_error_flag = False


if not (critical_error_flag or soft_error_flag):
    dbutils.notebook.exit("✅ Skipping QA checks: No critical or Non-Critical errors found in pre-validation.")
else:
    print("🚀 Running QA checks due to detected errors.")


gz_expected_info = dbutils.jobs.taskValues.get(taskKey="Pre_validation", key="gz_expected_info", debugValue=[])
gz_file_paths = dbutils.jobs.taskValues.get(taskKey="Pre_validation", key="gz_file_paths", debugValue={})

# Defensive check if gz_expected_info or gz_file_paths is empty
if not gz_expected_info or not gz_file_paths:
    dbutils.notebook.exit("⚠️ No gz_expected_info or gz_file_paths received from Pre-validation. QA cannot proceed.")

# Read reconciliation log table dynamically from widget
ctl_log_df = spark.table(reconciliation_table)

# Get the latest processed CTL file entry (assumed logic)
ctl_candidate_df = (
    ctl_log_df
    .orderBy(col("Processed_Timestamp").desc())
    .limit(1)
)

ctl_row = ctl_candidate_df.collect()
if not ctl_row:
    dbutils.notebook.exit(f"⚠️ No CTL file found for QA checks in table: {reconciliation_table}")

ctl_file_name = ctl_row[0]['CTL_File']
ctl_hash = ctl_row[0]['CTL_Hash']

print(f"📄 Selected CTL file for QA: {ctl_file_name}, Hash: {ctl_hash}")


missing_files = [name for name, _ in gz_expected_info if name not in gz_file_paths]
if missing_files:
    raise Exception(f"❌ Missing .gz files in gz_file_paths: {missing_files}")

# Get paths to read
gz_paths = [gz_file_paths[name] for name, _ in gz_expected_info]

# Read .gz files as DataFrame
try:
    df = spark.read.option("header", "true").csv(gz_paths)
except Exception as e:
    raise Exception(f"❌ Failed to read .gz files. Error: {str(e)}")


error_records = []
date_received = datetime.now()
start_load = date_received
end_load = datetime.now()

# Null checks
for col_name in columns_to_check:
    if col_name in df.columns:
        null_count = df.filter(col(col_name).isNull() | isnan(col(col_name))).count()
        if null_count > 0:
            for file_name, _ in gz_expected_info:
                error_records.append((file_name, str(date_received), str(start_load), str(end_load), null_count, f"{col_name} contains NULL values"))
    else:
        for file_name, _ in gz_expected_info:
            error_records.append((file_name, str(date_received), str(start_load), str(end_load), 0, f"{col_name} column is missing"))

# Empty file check
row_count = df.count()
if row_count == 0:
    for file_name, _ in gz_expected_info:
        error_records.append((file_name, str(date_received), str(start_load), str(end_load), 0, "File is empty"))


if error_records:
    error_schema = StructType([
        StructField("File_Name", StringType(), True),
        StructField("Date_Received", StringType(), True),
        StructField("Start_Load_Date", StringType(), True),
        StructField("End_Load_Date", StringType(), True),
        StructField("Row_Number", LongType(), True),
        StructField("Error_Description", StringType(), True)
    ])
    error_df = spark.createDataFrame(error_records, schema=error_schema)

    if error_table:
        error_df.write.mode("append").saveAsTable(error_table)
        print(f"✅ Error report written to table: {error_table}")
    else:
        print("⚠️ No error_table specified. Skipping write.")
else:
    print("✅ No errors found. QA passed.")
